## Imports and setup

Loads taxon lineages into `organism_utility.taxon` (Stage 4a). Taxa are bulk loaded with postgres `COPY`, one Linnean rank at a time (kingdom first): parents are resolved from the ranks already loaded, taxa already in the database are skipped and only new taxa are copied. Each load is one transaction and rerunning inserts nothing.

Input is excel or csv in one of two layouts:
- **single column**: one lineage per row, e.g. `k__Bacteria|p__Bacillota|...|g__Niallia|s__` (prefixes from `taxon_rank.prefix`), as in `eDNA/excel/taxa_singlecolumn.xlsx`
- **multi column**: one column per rank named `kingdom` ... `variety`, as in `eDNA/excel/taxa_multicolumn.xlsx`

Each taxon is linked to a taxonomy reference: the process parameters `taxonomy_reference` / `taxonomy_reference_version` set the default, an optional column of the same name in the file overrides it per row. The reference must exist in `organism_utility.taxonomy_reference`.

Species are stored with the epithet as `name` and the binomial as `scientific_name`.

In [ ]:
# Standard library imports
from os import path, getcwd

import sys

# Get the working directory of the notebook
notebook_path = getcwd()

# Set the path to the root directory of this jupyter notebook (python) project
module_path = path.abspath(path.join('../..'))

if module_path not in sys.path:

    sys.path.append(module_path)

# Package application imports
from src.lib import Initiate_process

# Import the custom class for running processes defined in the ai4sh package
from src.ai4sh import Run_process

## Scheme file
To run this cell you need to set the [scheme_file](https://xspatula.github.io/setup_core_db_docs/framework/scheme_file/), that defines the location (path) to the project and associated process files and your user. You can set the path to the scheme file with a relative path (relative to this notebook), in your home directory using tilde (~) or with an absolute path in the syntax of your operating system. 

In [ ]:
# Set the path to the scheme_file

scheme_file = '../scheme_ai4sh.json'

## Single step: insert prokaryote taxa (SILVA)

**process**: insert_tabular_data (target process manage_taxon)

**process_file**: eDNA/insert_process/insert_taxon_prokaryotes.json

In [ ]:
#%%script false --no-raise-error
process_file = 'import_data/eDNA/insert_process/insert_taxon_prokaryotes.json'

structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, process_file)

if structured_process_D is not None:
    Run_process(structured_process_D, scheme_params_D)

## Single step: insert fungal taxa (UNITE)

**process_file**: eDNA/insert_process/insert_taxon_fungi.json

In [ ]:
#%%script false --no-raise-error
process_file = 'import_data/eDNA/insert_process/insert_taxon_fungi.json'

structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, process_file)

if structured_process_D is not None:
    Run_process(structured_process_D, scheme_params_D)

## Dual step (alternative to the single step cells above)

Step 1 translates the source file to a canonical lineage csv (`taxon_lineage.csv`, one unique lineage per row) plus a `manage_taxon.json` process file, so the result can be inspected before loading. Step 2 runs `manage_taxon.json`, which loads the csv and also writes the csv source of each per-rank COPY (`taxon_copy_<rank_id>_<rank>.csv`) to the same folder.

Remove the `%%script false` line to run a cell.

In [ ]:
%%script false --no-raise-error
# Step 1: translate
for process_file in ['import_data/eDNA/process/translate_taxon_prokaryotes.json',
                     'import_data/eDNA/process/translate_taxon_fungi.json']:

    structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, process_file)
    
    if structured_process_D is not None:
        Run_process(structured_process_D, scheme_params_D)

In [ ]:
%%script false --no-raise-error
# Step 2: manage (COPY)
for process_file in ['import_data/eDNA/manage_process/taxon_prokaryotes/manage_taxon.json',
                     'import_data/eDNA/manage_process/taxon_fungi/manage_taxon.json']:

    structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, process_file)
    
    if structured_process_D is not None:
        Run_process(structured_process_D, scheme_params_D)